# Model retrained on ml/features.py-computed features

The training data here (`nigerian_transactions_v2_features.csv`) was generated
by running the raw dataset through `ml/features.py`'s `compute_features()` -
the exact same function Chidera's live backend calls - rather than using the
HuggingFace dataset's own pre-baked aggregate columns. This closes the
training/live parity gap noted in `docs/model-contract.md`.

Same evaluation approach as `05_gbm_baseline.ipynb`: held-out test, threshold
sweep, unseen-typology test - so results are directly comparable.

In [1]:
import pandas as pd

df = pd.read_csv('../data/processed/nigerian_transactions_v2_features.csv')
print(df.shape)
print(df['fraud_type'].value_counts(dropna=False))
print(df['is_fraud'].mean())

(1000002, 28)
fraud_type
NaN                        960981
Account Takeover            37009
Identity Fraud               2009
Impossible Travel Fraud         3
Name: count, dtype: int64
0.03902092195815608


In [2]:
from sklearn.model_selection import train_test_split

FEATURES = [
    "amount_ngn", "spending_deviation_score", "is_amount_unusual", "velocity_score",
    "txn_count_last_1h", "txn_count_last_24h", "total_amount_last_1h",
    "time_since_last", "avg_gap_between_txns",
    "user_txn_count_total", "user_avg_txn_amt", "user_std_txn_amt", "user_txn_frequency_24h",
    "txn_hour", "is_weekend", "is_salary_week", "is_night_txn",
    "device_seen_count", "is_device_shared", "is_ato_risk",
    "geo_anomaly_score", "geospatial_velocity_anomaly",
    "is_high_risk_state", "is_high_risk_lga",
]

X = df[FEATURES]
y = df['is_fraud']
fraud_type = df['fraud_type']

X_train, X_test, y_train, y_test, ft_train, ft_test = train_test_split(
    X, y, fraud_type, test_size=0.2, random_state=42, stratify=y
)
print("Train:", X_train.shape, " Fraud rate:", y_train.mean())
print("Test:", X_test.shape, " Fraud rate:", y_test.mean())

Train: (800001, 24)  Fraud rate: 0.03902120122349847
Test: (200001, 24)  Fraud rate: 0.039019804900975494


In [3]:
from sklearn.ensemble import HistGradientBoostingClassifier
import time

model = HistGradientBoostingClassifier(
    max_iter=200, max_depth=8, learning_rate=0.1,
    class_weight='balanced', random_state=42, verbose=1
)

start = time.time()
model.fit(X_train, y_train)
print(f"\nTraining took {time.time() - start:.1f} seconds")

Binning 0.138 GB of training data: 

0.611 s
Binning 0.015 GB of validation data: 

0.026 s
Fitting gradient boosted rounds:


Fit 12 trees in 2.985 s, (372 total leaves)
Time spent computing histograms: 0.391s
Time spent finding best splits:  0.019s
Time spent applying splits:      0.077s
Time spent predicting:           0.019s

Training took 3.1 seconds


## Held-out evaluation - threshold sweep

Same method as the v1 (HuggingFace-feature) model, for direct comparison.

In [4]:
from sklearn.metrics import precision_recall_curve, precision_score, recall_score, confusion_matrix
import numpy as np

y_proba = model.predict_proba(X_test)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)

f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)
best_idx = np.argmax(f1_scores)
print(f"Best F1 threshold: {thresholds[best_idx]:.4f}")
print(f"At that threshold - Precision: {precisions[best_idx]:.4f}, Recall: {recalls[best_idx]:.4f}")

print("\nThreshold sweep - precision, recall, and false-positive burden per 1,000:")
for t in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    preds_at_t = (y_proba >= t).astype(int)
    p = precision_score(y_test, preds_at_t, zero_division=0)
    r = recall_score(y_test, preds_at_t, zero_division=0)
    cm_t = confusion_matrix(y_test, preds_at_t)
    fp_t = cm_t[0][1]
    total_legit_t = cm_t[0].sum()
    burden = fp_t / total_legit_t * 1000
    n_alerts = preds_at_t.sum()
    print(f"Threshold {t}: precision={p:.3f}, recall={r:.3f}, FP burden={burden:.1f}/1000, total alerts={n_alerts}")

Best F1 threshold: 0.4975
At that threshold - Precision: 0.0394, Recall: 0.8693

Threshold sweep - precision, recall, and false-positive burden per 1,000:
Threshold 0.3: precision=0.039, recall=1.000, FP burden=1000.0/1000, total alerts=200001


Threshold 0.4: precision=0.039, recall=0.999, FP burden=999.1/1000, total alerts=199817
Threshold 0.5: precision=0.039, recall=0.336, FP burden=335.3/1000, total alerts=67078


Threshold 0.6: precision=0.000, recall=0.000, FP burden=0.0/1000, total alerts=0
Threshold 0.7: precision=0.000, recall=0.000, FP burden=0.0/1000, total alerts=0


Threshold 0.8: precision=0.000, recall=0.000, FP burden=0.0/1000, total alerts=0
Threshold 0.9: precision=0.000, recall=0.000, FP burden=0.0/1000, total alerts=0


In [5]:
from sklearn.inspection import permutation_importance

# HistGradientBoostingClassifier has no built-in feature_importances_,
# so use permutation importance on a subsample of the test set (faster).
sample_idx = X_test.sample(n=20000, random_state=42).index
result = permutation_importance(
    model, X_test.loc[sample_idx], y_test.loc[sample_idx],
    n_repeats=3, random_state=42, n_jobs=2
)
importances = pd.Series(result.importances_mean, index=FEATURES).sort_values(ascending=False)
print(importances)

avg_gap_between_txns           2.056667e-02
spending_deviation_score       1.195000e-02
user_std_txn_amt               6.916667e-03
user_avg_txn_amt               3.683333e-03
time_since_last                4.000000e-04
is_weekend                     1.333333e-04
is_salary_week                 8.333333e-05
velocity_score                 0.000000e+00
is_high_risk_state             0.000000e+00
geospatial_velocity_anomaly    0.000000e+00
is_ato_risk                    0.000000e+00
txn_count_last_1h              0.000000e+00
is_amount_unusual              0.000000e+00
user_txn_frequency_24h         0.000000e+00
is_device_shared               0.000000e+00
device_seen_count              0.000000e+00
total_amount_last_1h          -7.401487e-17
is_night_txn                  -1.833333e-04
amount_ngn                    -7.166667e-04
is_high_risk_lga              -1.166667e-03
txn_count_last_24h            -1.700000e-03
txn_hour                      -1.800000e-03
geo_anomaly_score             -3

## Unseen-typology test

In [6]:
def unseen_typology_test(df, features, holdout_type, random_state=42):
    is_holdout_fraud = df['fraud_type'] == holdout_type
    train_pool = df[~is_holdout_fraud]
    holdout_fraud_rows = df[is_holdout_fraud]

    X_train_ut = train_pool[features]
    y_train_ut = train_pool['is_fraud']

    model_ut = HistGradientBoostingClassifier(
        max_iter=200, max_depth=8, learning_rate=0.1,
        class_weight='balanced', random_state=random_state
    )
    model_ut.fit(X_train_ut, y_train_ut)

    X_holdout = holdout_fraud_rows[features]
    y_holdout_pred = model_ut.predict(X_holdout)

    recall = y_holdout_pred.mean()
    n = len(holdout_fraud_rows)
    n_caught = y_holdout_pred.sum()

    print(f"=== Unseen-typology test: {holdout_type} ===")
    print(f"Held-out fraud examples (never in training): {n}")
    print(f"Caught by model trained WITHOUT this pattern: {n_caught} ({recall:.4f} recall)")
    print()
    return recall

recall_at = unseen_typology_test(df, FEATURES, 'Account Takeover')
recall_if = unseen_typology_test(df, FEATURES, 'Identity Fraud')

print("Summary:")
print(f"Account Takeover (unseen): {recall_at:.4f}")
print(f"Identity Fraud (unseen):   {recall_if:.4f}")
print("Impossible Travel Fraud: SKIPPED - too few examples in this sample")

=== Unseen-typology test: Account Takeover ===
Held-out fraud examples (never in training): 37009
Caught by model trained WITHOUT this pattern: 18269 (0.4936 recall)



=== Unseen-typology test: Identity Fraud ===
Held-out fraud examples (never in training): 2009
Caught by model trained WITHOUT this pattern: 550 (0.2738 recall)

Summary:
Account Takeover (unseen): 0.4936
Identity Fraud (unseen):   0.2738
Impossible Travel Fraud: SKIPPED - too few examples in this sample


In [7]:
import joblib
joblib.dump(model, '../models/fraud_model_v2.pkl')
print("Saved models/fraud_model_v2.pkl")

Saved models/fraud_model_v2.pkl
